In [44]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('join practice').getOrCreate()

In [45]:
from google.colab import files

uploaded_01 = files.upload()
uploaded_02 = files.upload()
uploaded_03 = files.upload()

Saving patients.csv to patients.csv


Saving appointments.csv to appointments.csv


Saving patient_preferences.json to patient_preferences.json


In [46]:
patients_df = spark.read.csv(
    'patients.csv',
    header = True,
    inferSchema= True
)

appointments_df = spark.read.csv(
    'appointments.csv',
    header = True,
    inferSchema = True
)

patient_preferences_df = spark.read.option(
    'multiline',
    'true'
).json('patient_preferences.json')

In [47]:
patients_df.show()
appointments_df.show()
patient_preferences_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [48]:
patients_df.printSchema()
print('count patients : ', patients_df.count())
print('1 -> 5 rows')
patients_df.show(5)
patients_df.select(col('city').alias('distinct city')).distinct().show()
appointments_df.select(col('department').alias('distinct dept')).distinct().show()
patients_df.write.mode('overwrite').parquet('patients.parquet')

par_rec = spark.read.parquet('patients.parquet')

par_rec_count = par_rec.count()
csv_rec_count = patients_df.count()
print('count comp :', par_rec_count, csv_rec_count)

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)

count patients :  10
1 -> 5 rows
+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+-------

In [49]:
patients_df.filter(col('city') == 'Hyderabad').show()
patients_df.filter(col('gender') == 'Female').show()
patients_df.filter(col('age') > 40).show()
appointments_df.filter(col('status') == 'Complete').show()
appointments_df.filter(col('status') == 'Pending').show()
appointments_df.filter(col('consultation_fee') > 1500).show()
patients_df.filter(col('insurance_status') == 'Active').show()
patients_df.filter(col('insurance_status') == 'Inactive').show()
patients_df.filter(col('blood_group') == 'O+').show()
appointments_df.filter(col('department') == 'Cardiology').show()


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110

In [50]:
appointments_df.columns

['appointment_id',
 'patient_id',
 'doctor_name',
 'department',
 'appointment_date',
 'consultation_fee',
 'status']

In [51]:
patients_df.columns

['patient_id',
 'patient_name',
 'city',
 'age',
 'gender',
 'blood_group',
 'insurance_status']

In [52]:
patient_preferences_df.columns

['contact', 'patient_id', 'preferred_hospital']

In [53]:
patients_df.filter(col('city').isNull()).show()
patients_df.filter(col('blood_group').isNull()).show()
appointments_df.filter(col('consultation_fee').isNull()).show()

print('null count: ')
for c in patients_df.columns:
  print(c, end=" : ")
  print(patients_df.filter(col(c).isNull()).count())

f_patients_df = patients_df.na.fill({
    'city': 'Unknown',
    'blood_group': 'Not Available'
})

f_appointments_df = appointments_df.na.fill({
    'consultation_fee': 0
})

appointments_df.na.drop(subset=['consultation_fee']).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+-------

In [54]:
from functools import reduce

condition = reduce(
    lambda a,b : a|b,
    [col(c).isNull() for c in patients_df.columns]
)

q_patients_df = patients_df.withColumn(
    'data_quality_status',
    when(condition, 'Incomplete')
    .otherwise('Complete')
)

no_q_patients_df_com = q_patients_df.filter(col('data_quality_status') == 'Complete').count()
no_q_patients_df_incom = q_patients_df.filter(col('data_quality_status') == 'Incomplete').count()
print('compare com : incom', no_q_patients_df_com, no_q_patients_df_incom)

compare com : incom 8 2


In [55]:
patients_df.select(upper('patient_name')).show()
patients_df.select(lower('patient_name')).show()
patients_df.select('patient_name', length('patient_name').alias('len(name)')).show()
patients_df.select(col('patient_name'), substring('patient_name', 1, 3).alias('name[:3]')).show()

+-------------------+
|upper(patient_name)|
+-------------------+
|       RAHUL SHARMA|
|        PRIYA REDDY|
|         AMIT KUMAR|
|        SNEHA PATEL|
|         FARHAN ALI|
|         NEHA SINGH|
|        ARJUN VERMA|
|         MEERA NAIR|
|          KIRAN RAO|
|        NISHA REDDY|
+-------------------+

+-------------------+
|lower(patient_name)|
+-------------------+
|       rahul sharma|
|        priya reddy|
|         amit kumar|
|        sneha patel|
|         farhan ali|
|         neha singh|
|        arjun verma|
|         meera nair|
|          kiran rao|
|        nisha reddy|
+-------------------+

+------------+---------+
|patient_name|len(name)|
+------------+---------+
|Rahul Sharma|       12|
| Priya Reddy|       11|
|  Amit Kumar|       10|
| Sneha Patel|       11|
|  Farhan Ali|       10|
|  Neha Singh|       10|
| Arjun Verma|       11|
|  Meera Nair|       10|
|   Kiran Rao|        9|
| Nisha Reddy|       11|
+------------+---------+

+------------+--------+
|patien

In [56]:
patients_df.withColumns({
    'age_group':
    when(col('age') < 18, 'minor')
    .otherwise('adult'),
    'insurance_flag':
    when(col('insurance_status') == 'Active', True)
    .otherwise(False),
    'senior_citizen':
    when(col('age') > 60, True)
    .otherwise(False)
}).show()

+----------+------------+---------+---+------+-----------+----------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|    adult|          true|         false|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|    adult|          true|         false|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|    adult|         false|         false|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|    adult|          true|         false|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|    adult|          true|         false|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inacti

In [57]:
patients_df.select(concat(col('patient_name'), lit(' : '), col('city'))).show()
patients_df.select(trim(col('patient_name')).alias('trimed name')).show()
patients_df.withColumn('city', upper('city')).show()

+-------------------------------+
|concat(patient_name,  : , city)|
+-------------------------------+
|           Rahul Sharma : Hy...|
|           Priya Reddy : Ban...|
|            Amit Kumar : Mumbai|
|           Sneha Patel : Che...|
|             Farhan Ali : Delhi|
|                           NULL|
|             Arjun Verma : Pune|
|             Meera Nair : Kochi|
|           Kiran Rao : Hyder...|
|           Nisha Reddy : Ban...|
+-------------------------------+

+------------+
| trimed name|
+------------+
|Rahul Sharma|
| Priya Reddy|
|  Amit Kumar|
| Sneha Patel|
|  Farhan Ali|
|  Neha Singh|
| Arjun Verma|
|  Meera Nair|
|   Kiran Rao|
| Nisha Reddy|
+------------+

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|         O+|          Active|
|       

In [58]:
patients_df.groupBy('city').agg(count('*').alias('no.patients / city')).show()
patients_df.groupBy('gender').agg(count('*').alias('no.patients / gender')).show()
patients_df.groupBy('blood_group').agg(count('*').alias('no.patients / blood')).show()
appointments_df.groupBy('department').agg(count('*').alias('appoint / dept')).show()
patients_df.groupBy('city').agg(avg('age').alias('avg age')).show()
patients_df.groupBy('city').agg(max('age')).show()
patients_df.groupBy('city').agg(min('age')).show()
appointments_df.groupBy('department').agg(avg('consultation_fee')).show()
appointments_df.groupBy('department').agg(sum('consultation_fee')).show()
appointments_df.groupBy('department').agg(sum('consultation_fee').alias('sum_consult')).orderBy(col('sum_consult').desc()).show(1)

+---------+------------------+
|     city|no.patients / city|
+---------+------------------+
|Bangalore|                 2|
|    Kochi|                 1|
|  Chennai|                 1|
|     NULL|                 1|
|   Mumbai|                 1|
|     Pune|                 1|
|    Delhi|                 1|
|Hyderabad|                 2|
+---------+------------------+

+------+--------------------+
|gender|no.patients / gender|
+------+--------------------+
|Female|                   5|
|  Male|                   5|
+------+--------------------+

+-----------+-------------------+
|blood_group|no.patients / blood|
+-----------+-------------------+
|        AB+|                  1|
|       NULL|                  1|
|         O+|                  2|
|         O-|                  1|
|         B+|                  2|
|         A+|                  3|
+-----------+-------------------+

+-----------+--------------+
| department|appoint / dept|
+-----------+--------------+
|  Neurology|     

In [60]:
patients_df.join(
    patient_preferences_df,
    'patient_id',
    'inner'
).show()

patients_df.join(
    patient_preferences_df,
    'patient_id',
    'left'
).show()

patients_df.join(
    patient_preferences_df,
    'patient_id',
    'right'
).show()

patients_df.join(
    patient_preferences_df,
    'patient_id',
    'full'
).show()

+----------+------------+---------+---+------+-----------+----------------+--------------------+------------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|             contact|preferred_hospital|
+----------+------------+---------+---+------+-----------+----------------+--------------------+------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|{rahul@gmail.com,...|            Apollo|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|{priya@gmail.com,...|           Yashoda|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|  {NULL, 9876500013}|              Care|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|{sneha@gmail.com,...|              NULL|
+----------+------------+---------+---+------+-----------+----------------+--------------------+------------------+

+----------+------------+---------+---+------+-----------+-------------

In [63]:
patients_df.join(
    appointments_df,
    'patient_id',
    'left'
).filter(col('appointment_id').isNull()).show()

patients_df.join(
    appointments_df,
    'patient_id',
    'right'
).filter(col('patient_name').isNull()).show()

patients_df.join(
    appointments_df,
    'patient_id',
    'full'
).groupBy('patient_id').agg(
    count(col('appointment_id')).alias('appoint count')
).show()

patients_df.join(
    appointments_df,
    'patient_id',
    'full'
).groupBy('patient_id').agg(
    sum(col('consultation_fee')).alias('total fee')
).show()

patients_df.join(
    appointments_df,
    'patient_id',
    'full'
).groupBy('patient_id').agg(
    sum(col('consultation_fee')).alias('total fee')
).orderBy(col('total fee').desc()).show(1)

patients_df.join(
    appointments_df,
    'patient_id',
    'full'
).groupBy('patient_id').agg(
    count(col('appointment_id')).alias('appoint count')
).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name|department|appointment_date|consultation_fee|status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|          NULL|       NULL|      NULL|            NULL|            NULL|  NULL|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|          NULL|       NULL|      NULL|            NULL|            NULL|  NULL|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+

+----------+------------+----+----+------+-----------+---------------

In [70]:
full_patients_df = patients_df.join(
    appointments_df.groupby('patient_id').agg(sum('consultation_fee').alias('total_fee')),
    'patient_id',
    'left'
)

In [73]:
from pyspark.sql.window import Window as win

win_spec = win.orderBy(col('total_fee').desc())

full_patients_df.withColumn(
    'rank',
    rank().over(win_spec)
).show()

+----------+------------+---------+---+------+-----------+----------------+---------+----+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|rank|
+----------+------------+---------+---+------+-----------+----------------+---------+----+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|     2500|   1|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|     2500|   1|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|     2500|   1|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|     2000|   4|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|     2000|   4|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|     1500|   6|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     1000|   7|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|     NULL|   8|

In [74]:
from pyspark.sql.window import Window as win

win_spec = win.orderBy(col('total_fee').desc())

full_patients_df.withColumn(
    'dense_rank',
    dense_rank().over(win_spec)
).show()

+----------+------------+---------+---+------+-----------+----------------+---------+----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|dense_rank|
+----------+------------+---------+---+------+-----------+----------------+---------+----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|     2500|         1|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|     2500|         1|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|     2500|         1|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|     2000|         2|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|     2000|         2|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|     1500|         3|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     1000|         4|
|       106|  Neha Singh|     

In [75]:
from pyspark.sql.window import Window as win

win_sec = win.orderBy(col('total_fee'))

full_patients_df.withColumn(
    'row num',
    row_number().over(win_spec)
).show()

+----------+------------+---------+---+------+-----------+----------------+---------+-------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|row num|
+----------+------------+---------+---+------+-----------+----------------+---------+-------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|     2500|      1|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|     2500|      2|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|     2500|      3|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|     2000|      4|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|     2000|      5|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|     1500|      6|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     1000|      7|
|       106|  Neha Singh|     NULL| 38|Female|         A+|  

In [79]:
from pyspark.sql.window import Window as win

win_spec = win.orderBy(col('total_fee').desc())

full_patients_df.withColumn(
    'dense_rank',
    dense_rank().over(win_spec)
).filter(col('dense_rank') == 1).show()

full_patients_df.withColumn(
    'dense_rank',
    dense_rank().over(win_spec)
).filter(col('dense_rank').isin(1, 2, 3)).show()

+----------+------------+---------+---+------+-----------+----------------+---------+----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|dense_rank|
+----------+------------+---------+---+------+-----------+----------------+---------+----------+
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|     2500|         1|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|     2500|         1|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|     2500|         1|
+----------+------------+---------+---+------+-----------+----------------+---------+----------+

+----------+------------+---------+---+------+-----------+----------------+---------+----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|dense_rank|
+----------+------------+---------+---+------+-----------+----------------+---------+----------+
|       104| Sneha Patel|  Ch

In [82]:
full_patients_df = full_patients_df.na.fill({
    'total_fee': 0
})

win_spec = win.partitionBy('city').orderBy(col('total_fee').desc())
full_patients_df.withColumn(
    'row_no',
    row_number().over(win_spec)
).filter(col('row_no') == 1).drop('row_no').show()

+----------+------------+---------+---+------+-----------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|
+----------+------------+---------+---+------+-----------+----------------+---------+
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|        0|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|     2500|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|     2500|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     1000|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|     2500|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|        0|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|     1500|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|     2000|
+----------+------------+---------+---+------+--------

In [83]:
win_spec = win.partitionBy('city').orderBy('total_fee')
full_patients_df.withColumn(
    'row_no',
    row_number().over(win_spec)
).filter(col('row_no') == 1).drop('row_no').show()

+----------+------------+---------+---+------+-----------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|
+----------+------------+---------+---+------+-----------+----------------+---------+
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|        0|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|     2000|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|     2500|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     1000|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|        0|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|        0|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|     1500|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|     2000|
+----------+------------+---------+---+------+--------

In [84]:
win_spec = win.orderBy('patient_id')

full_patients_df.withColumn(
    'running sum',
    sum('total_fee').over(win_spec)
).show()

+----------+------------+---------+---+------+-----------+----------------+---------+-----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|total_fee|running sum|
+----------+------------+---------+---+------+-----------+----------------+---------+-----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|     2500|       2500|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|     2000|       4500|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|     1500|       6000|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|     2500|       8500|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     1000|       9500|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|        0|       9500|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|     2000|      11500|
|       108|  Meera 

In [86]:
win_spec = win.orderBy('appointment_date')

appointments_df.withColumns({
    'prev fee': lag('consultation_fee').over(win_spec),
    'next fee': lead('consultation_fee').over(win_spec)
}).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+--------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|prev fee|next fee|
+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+--------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|    NULL|    2000|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|    1500|    1000|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|    2000|    1500|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|    1000|    1000|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|    1500|    2000|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Com

In [87]:
patient_preferences_df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [91]:
flat_patient_prefference_df = patient_preferences_df.select(
    col('patient_id'),
    col('preferred_hospital'),
    col('contact.email').alias('email'),
    col('contact.phone').alias('phone')
)
flat_patient_prefference_df.columns

['patient_id', 'preferred_hospital', 'email', 'phone']

In [92]:
flat_patient_prefference_df.filter(col('phone').isNull()).show()
flat_patient_prefference_df.filter(col('email').isNull()).show()
flat_patient_prefference_df.filter(col('preferred_hospital').isNull()).show()
flat_patient_prefference_df.na.fill({
    'phone': 0000000000,
    'email': 'example@email.com',
    'preferred_hospital': 'Not Given'
})


+----------+------------------+---------------+-----+
|patient_id|preferred_hospital|          email|phone|
+----------+------------------+---------------+-----+
|       102|           Yashoda|priya@gmail.com| NULL|
+----------+------------------+---------------+-----+

+----------+------------------+-----+----------+
|patient_id|preferred_hospital|email|     phone|
+----------+------------------+-----+----------+
|       103|              Care| NULL|9876500013|
+----------+------------------+-----+----------+

+----------+------------------+---------------+----------+
|patient_id|preferred_hospital|          email|     phone|
+----------+------------------+---------------+----------+
|       104|              NULL|sneha@gmail.com|9876500014|
+----------+------------------+---------------+----------+



DataFrame[patient_id: bigint, preferred_hospital: string, email: string, phone: string]

In [93]:
patient_join_preferrence = patients_df.join(
    flat_patient_prefference_df,
    'patient_id',
    'left'
)
patient_join_preferrence.show()

+----------+------------+---------+---+------+-----------+----------------+------------------+---------------+----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|preferred_hospital|          email|     phone|
+----------+------------+---------+---+------+-----------+----------------+------------------+---------------+----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|            Apollo|rahul@gmail.com|9876500011|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Yashoda|priya@gmail.com|      NULL|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|              Care|           NULL|9876500013|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|              NULL|sneha@gmail.com|9876500014|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|              NULL|           NULL|      NULL|
|       106|  Neha Singh

In [94]:
patient_preferences_df.show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{rahul@gmail.com,...|       101|            Apollo|
|{priya@gmail.com,...|       102|           Yashoda|
|  {NULL, 9876500013}|       103|              Care|
|{sneha@gmail.com,...|       104|              NULL|
+--------------------+----------+------------------+



In [95]:
patients_df.createOrReplaceTempView('patients')
appointments_df.createOrReplaceTempView('appointments')
spark.catalog.listTables()

[Table(name='appointments', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='patients', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [104]:
spark.sql('select * from patients;').show()
spark.sql("select * from patients where city = 'Hyderabad';").show()
spark.sql("""
  select city, count(patient_id) as patient_count from patients
  group by city;
""").show()
spark.sql("""
  select department, count(appointment_id) as appointment_count from appointments
  group by department;
""").show()
spark.sql("""
  select department, avg(consultation_fee) from appointments
  group by department;
""").show()
spark.sql("""
  select max(consultation_fee) from appointments;
""").show()
spark.sql("""
  select patient_id, count(*) as appointment_count from appointments
  group by patient_id;
""").show()
spark.sql("""
with top_patients as (
    select patient_id
    from appointments
    group by patient_id
    order by sum(consultation_fee) desc
    limit 5
)

select *
from patients
where patient_id in (
    select patient_id
    from top_patients
)
""").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [105]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

patient_preferences_df = spark.read.option(
    "multiline",
    "true"
).json("patient_preferences.json")

In [106]:
patients_df = patients_df.fillna({
    "city": "Unknown",
    "blood_group": "Unknown"
})

appointments_df = appointments_df.fillna({
    "consultation_fee": 0
})

In [107]:
full_df = patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).join(
    patient_preferences_df,
    "patient_id",
    "left"
)

In [108]:
full_df = full_df.withColumn(
    "age_group",
    when(col("age") < 18, "Child")
    .when(col("age") < 40, "Adult")
    .when(col("age") < 60, "Middle Age")
    .otherwise("Senior")
)

In [109]:
full_df = full_df.withColumn(
    "revenue_category",
    when(col("consultation_fee") >= 2000, "High")
    .otherwise("Normal")
)

In [110]:
patient_spending_df = appointments_df.groupBy("patient_id").agg(
    sum("consultation_fee").alias("total_spending")
)
patient_spending_df.show()

+----------+--------------+
|patient_id|total_spending|
+----------+--------------+
|       108|             0|
|       101|          2500|
|       103|          1500|
|       120|          1500|
|       107|          2000|
|       102|          2000|
|       105|          1000|
|       110|          2500|
|       104|          2500|
+----------+--------------+



In [111]:
department_revenue_df = appointments_df.groupBy("department").agg(
    sum("consultation_fee").alias("department_revenue")
)

department_revenue_df.show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|  Neurology|              4000|
|Dermatology|              2000|
| Cardiology|              4500|
|Orthopedics|              5000|
+-----------+------------------+



In [112]:
full_df.write.mode("overwrite").parquet("hospital_analytics.parquet")

In [113]:
print("Total Patients :", patients_df.count())

print("Total Appointments :", appointments_df.count())

print("Patient-wise Spending")
patient_spending_df.show()

print("Department Revenue")
department_revenue_df.show()

print("Appointments by Status")
appointments_df.groupBy("status").count().show()

print("Patients by City")
patients_df.groupBy("city").count().show()

Total Patients : 10
Total Appointments : 10
Patient-wise Spending
+----------+--------------+
|patient_id|total_spending|
+----------+--------------+
|       108|             0|
|       101|          2500|
|       103|          1500|
|       120|          1500|
|       107|          2000|
|       102|          2000|
|       105|          1000|
|       110|          2500|
|       104|          2500|
+----------+--------------+

Department Revenue
+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|  Neurology|              4000|
|Dermatology|              2000|
| Cardiology|              4500|
|Orthopedics|              5000|
+-----------+------------------+

Appointments by Status
+---------+-----+
|   status|count|
+---------+-----+
|Completed|    7|
|Cancelled|    1|
|  Pending|    2|
+---------+-----+

Patients by City
+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai